# Technip Energies India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** hcxg.fa.em2.oraclecloud.com — Oracle Cloud HCM

**ATS:** Oracle Cloud HCM (Candidate Experience)

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path
SCRIPTS_DIR = Path.home() / 'Job_Scrapers' / 'All_Scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from scraper_utils import *
from bs4 import BeautifulSoup
from datetime import datetime
import requests
LOCATION_FILTER = 'India'
print('Imports loaded. Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-01 00:55:52


In [3]:
COMPANY = 'Technip_Energies'
OUTPUT_DIR = get_output_dir(COMPANY)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Technip_Energies/Outputs/2026_04_01


In [4]:
print('=' * 60)
print('TECHNIP ENERGIES INDIA JOB SCRAPER')
print('ATS: Oracle Cloud HCM (hcxg.fa.em2.oraclecloud.com)')
print('=' * 60)

ORACLE_BASE = 'https://hcxg.fa.em2.oraclecloud.com'
REST_API = f'{ORACLE_BASE}/hcmRestApi/resources/latest/recruitingCEJobRequisitions'
UI_BASE = f'{ORACLE_BASE}/hcmUI/CandidateExperience/en/sites/CX_1/requisitions'

session = get_session()
session.headers.update({'Accept': 'application/json', 'Content-Type': 'application/json'})

technip_jobs = []
seen_ids = set()

# Oracle Cloud HCM REST API approach
print('  Trying Oracle HCM REST API...')
offset = 0
limit = 25
try:
    while offset < 500:
        params = {
            'expand': 'requisitionList.secondaryLocations',
            'onlyData': 'true',
            'limit': limit,
            'offset': offset,
        }
        if LOCATION_FILTER:
            params['q'] = f'PrimaryLocation.CountryName="{LOCATION_FILTER}"'
        resp = session.get(REST_API, params=params, timeout=30)
        if resp.status_code != 200:
            print(f'  [ERROR] REST API HTTP {resp.status_code}')
            break
        data = resp.json()
        items = data.get('items', data.get('requisitionList', []))
        total = data.get('totalResults', data.get('count', 0))
        if offset == 0: print(f'  Total requisitions: {total}')
        if not items: break
        print(f'  offset={offset}: {len(items)} jobs')
        for item in items:
            req = item if 'Title' in item else item.get('requisitionList', item)
            title = req.get('Title', req.get('title', ''))
            if not is_valid_job_title(title): continue
            req_id = str(req.get('Id', req.get('id', req.get('RequisitionNumber', ''))))
            loc = req.get('PrimaryLocation', {}) if isinstance(req.get('PrimaryLocation'), dict) else {}
            city = loc.get('Name', loc.get('CityName', 'India')).split(',')[0].strip() if loc else 'India'
            country = loc.get('CountryName', 'India') if loc else 'India'
            if LOCATION_FILTER and LOCATION_FILTER.lower() not in country.lower(): continue
            dept = req.get('OrganizationName', req.get('BusinessUnit', ''))
            posted = req.get('PostedDate', req.get('postedDate', ''))
            date_posted = posted[:10] if posted else datetime.now().strftime('%Y-%m-%d')
            job_url = f"{UI_BASE}/job/{req_id}" if req_id else ''
            if req_id not in seen_ids:
                seen_ids.add(req_id)
                technip_jobs.append({
                    'job_id': req_id, 'title': title, 'company_name': 'Technip Energies',
                    'job_url': job_url, 'source_api_url': REST_API,
                    'business_unit': dept or '', 'raw_jd_text': req.get('ExternalDescriptionStr', ''),
                    'location_city': city, 'location_country': country,
                    'industry': 'Energy / Engineering / EPC', 'date_posted': date_posted,
                    'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Oracle Cloud HCM'
                })
        if offset + limit >= total: break
        offset += limit
        time.sleep(random.uniform(0.5, 1.5))
except Exception as e:
    print(f'  REST API error: {e}')

# Fallback: Selenium on the UI
if len(technip_jobs) < 3:
    print('\n  Falling back to Selenium on Oracle HCM UI...')
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    driver = setup_selenium()
    try:
        driver.get(UI_BASE)
        time.sleep(8)
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, '[class*="job"], a[href*="/job/"]')))
        except: time.sleep(5)
        for page in range(20):
            soup = BeautifulSoup(driver.page_source, 'lxml')
            cards = (soup.select('[class*="job-requisition"]') or soup.select('[class*="requisition"]') or
                     soup.select('[class*="job-card"]') or soup.select('[class*="result-item"]'))
            if not cards:
                links = soup.select('a[href*="/requisitions/job/"]')
                seen = set()
                for link in links:
                    p = link.find_parent(['li', 'div', 'article'])
                    if p and id(p) not in seen: cards.append(p); seen.add(id(p))
            new_count = 0
            for card in cards:
                title_el = (card.select_one('[class*="title"] a') or card.select_one('h3 a') or
                            card.select_one('a[href*="/job/"]'))
                title = title_el.get_text(strip=True) if title_el else ''
                if not is_valid_job_title(title): continue
                href = title_el.get('href', '') if title_el else ''
                job_url = href if href.startswith('http') else (ORACLE_BASE + href if href else '')
                job_id = href.rstrip('/').split('/')[-1] if href else str(abs(hash(title)))
                loc_el = card.select_one('[class*="location"]')
                loc = loc_el.get_text(strip=True) if loc_el else 'India'
                if LOCATION_FILTER and LOCATION_FILTER.lower() not in loc.lower(): continue
                if job_id not in seen_ids:
                    seen_ids.add(job_id)
                    technip_jobs.append({
                        'job_id': job_id, 'title': title, 'company_name': 'Technip Energies',
                        'job_url': job_url, 'source_api_url': UI_BASE, 'business_unit': '',
                        'raw_jd_text': card.get_text(' ', strip=True),
                        'location_city': loc.split(',')[0].strip(), 'location_country': 'India',
                        'industry': 'Energy / Engineering / EPC',
                        'date_posted': datetime.now().strftime('%Y-%m-%d'),
                        'is_active': True, 'salary_currency': 'INR', 'source_platform': 'Oracle Cloud HCM'
                    })
                    new_count += 1
            print(f'  Page {page+1}: {new_count} new jobs (total: {len(technip_jobs)})')
            if new_count == 0 and page > 0: break
            try:
                btn = driver.find_element(By.CSS_SELECTOR, 'a[aria-label*="Next"],button[aria-label*="Next"]')
                driver.execute_script('arguments[0].click();', btn)
                time.sleep(3)
            except: break
    except Exception as e:
        print(f'  Selenium error: {e}')
    finally:
        driver.quit()

print(f'\nTotal Technip Energies India jobs: {len(technip_jobs)}')

TECHNIP ENERGIES INDIA JOB SCRAPER
ATS: Oracle Cloud HCM (hcxg.fa.em2.oraclecloud.com)
  Trying Oracle HCM REST API...


  [ERROR] REST API HTTP 400

  Falling back to Selenium on Oracle HCM UI...


  Page 1: 0 new jobs (total: 0)

Total Technip Energies India jobs: 0


In [5]:
df_te = save_results(technip_jobs, 'Technip_Energies', OUTPUT_DIR)
if df_te is not None:
    cols = ['title','location_city','seniority_level','business_unit','job_url']
    cols = [c for c in cols if c in df_te.columns]
    print(df_te[cols].head(10).to_string())

  [WARN] No jobs found for Technip_Energies
